# AWQ Quantization + Trajectory Collection — Qwen2.5-3B
Two parts:
1. Quantize the 3B model to 4-bit AWQ for faster experiments
2. Collect generation trajectories from both BF16 and AWQ versions (saved to Drive)

After running: download `qwen3b_awq.zip`, then trajectories are on Drive for comparison.

In [ ]:
# Install deps
!pip install -q autoawq transformers torch datasets

import os, glob, zipfile, time, gc, torch, re
import numpy as np
from awq import AutoAWQForCausalLM
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from google.colab import drive

## Part 1: AWQ Quantization

In [ ]:
MODEL = "Qwen/Qwen2.5-3B-Instruct"
SAVE_DIR = "qwen3b_awq"
W_BIT = 4
GROUP_SIZE = 128
CALIB_SAMPLES = 128
print(f"Quantizing {MODEL} to {W_BIT}-bit AWQ...", flush=True)

In [ ]:
print("Loading model...", flush=True)
model = AutoAWQForCausalLM.from_pretrained(MODEL, device_map="cuda")
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB", flush=True)

In [ ]:
print("Quantizing (~20 min)...", flush=True)
model.quantize(
    tokenizer,
    quant_config={"zero_point": True, "q_group_size": GROUP_SIZE,
                  "w_bit": W_BIT, "version": "GEMM"},
    calib_data="pileval",
    max_calib_samples=CALIB_SAMPLES,
)
print("Quantization complete!", flush=True)

In [ ]:
print(f"Saving to {SAVE_DIR}...", flush=True)
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_quantized(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
!du -sh $SAVE_DIR

In [ ]:
# Zip for download
ZIP_NAME = f"{SAVE_DIR}.zip"
with zipfile.ZipFile(ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(SAVE_DIR):
        for f in files:
            zf.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), SAVE_DIR))
print(f"Zipped: {os.path.getsize(ZIP_NAME)/1e6:.0f}MB")
from google.colab import files
files.download(ZIP_NAME)

## Part 2: Trajectory Collection

Collects generation trajectories from both:
- **BF16** (original model, full precision) → Drive: `trajs_3b_bf16/`
- **AWQ** (quantized model) → Drive: `trajs_3b_awq/`

Trajectories per model: ~500 problems from GSM8K (saved in batches of 50).

In [ ]:
# Mount Drive
drive.mount('/content/drive')
DRIVE_BASE = "/content/drive/MyDrive/TrimTab/"
N_TRAJS = 500  # total trajectories per model
BATCH_SIZE_PROBLEMS = 50  # problems before saving a batch file
MAX_NEW = 100

N_LAYERS = 36
D_MODEL = 2048

def extract_trajectories(input_ids, gen_len, hidden_states, n_layers):
    plen = input_ids.shape[1]
    if gen_len <= 1: return None, None
    hs_at = torch.stack(
        [h[0, plen:plen+gen_len].float() for h in hidden_states[:n_layers]], dim=0
    )
    h_seq = hs_at[:, :-1]
    h_next = hs_at[:, 1:]
    v = h_next - h_seq
    return h_seq.permute(1, 0, 2), v.permute(1, 0, 2)

ds = load_dataset("openai/gsm8k", "main", split="test")

In [ ]:
def collect_trajectories(model, tokenizer, label, n_total=N_TRAJS):
    """Collect N trajectories from model, save to Drive in batches."""
    save_dir = os.path.join(DRIVE_BASE, f"trajs_3b_{label}/")
    os.makedirs(save_dir, exist_ok=True)

    buf_h, buf_v, batch_idx = [], [], 0
    count, i = 0, 0
    t0 = time.time()

    while count < n_total:
        prob = ds[i]
        msgs = [{"role": "user", "content": f"Q: {prob['question']}\nA:"}]
        inp = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                             return_tensors="pt").to(model.device)
        plen = inp.shape[1]
        out = model.generate(inp, max_new_tokens=MAX_NEW, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
        glen = out.shape[1] - plen
        i += 1
        if glen <= 1:
            continue

        with torch.no_grad():
            fwd = model(out, use_cache=False, output_hidden_states=True)
        h_seq, v_tgt = extract_trajectories(inp, glen, fwd.hidden_states, N_LAYERS)
        if h_seq is None:
            continue

        buf_h.append(h_seq.cpu().half())
        buf_v.append(v_tgt.cpu().half())
        count += 1

        if len(buf_h) >= BATCH_SIZE_PROBLEMS or count >= n_total:
            fname = os.path.join(save_dir, f"batch_{batch_idx:04d}.pt")
            torch.save({"hidden_seqs": torch.cat(buf_h),
                       "velocity_targets": torch.cat(buf_v) }, fname)
            buf_h, buf_v = [], []
            batch_idx += 1
            speed = count / (time.time() - t0) * 60
            print(f"  [{label}] [{count}/{n_total}] saved {fname} ({speed:.0f} trajs/min)", flush=True)

    with open(os.path.join(save_dir, "done.flag"), "w") as f:
        f.write(f"n_trajectories={count}\n")
    print(f"[{label}] Done — {count} trajectories in {batch_idx} batches", flush=True)

In [ ]:
# Clean up AWQ model to free VRAM
del model
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {torch.cuda.memory_allocated()/1e9:.1f}GB", flush=True)

In [ ]:
# ─── Trajectory Collection: BF16 model ───
print("Loading BF16 model...", flush=True)
model_bf16 = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct", torch_dtype=torch.bfloat16, device_map="cuda",
    trust_remote_code=True)
model_bf16.eval()
tok_bf16 = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct", trust_remote_code=True)
tok_bf16.pad_token = tok_bf16.eos_token
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB", flush=True)

collect_trajectories(model_bf16, tok_bf16, "bf16", N_TRAJS)

del model_bf16, tok_bf16
gc.collect(); torch.cuda.empty_cache()
print(f"\nBF16 done. VRAM free: {torch.cuda.memory_allocated()/1e9:.1f}GB", flush=True)

In [ ]:
# ─── Trajectory Collection: AWQ model ───
print("Loading AWQ model...", flush=True)
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
model_awq = AutoModelForCausalLM.from_pretrained(
    SAVE_DIR, trust_remote_code=True, device_map="cuda")
model_awq.eval()
tok_awq = AutoTokenizer.from_pretrained(SAVE_DIR, trust_remote_code=True)
tok_awq.pad_token = tok_awq.eos_token
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB", flush=True)

collect_trajectories(model_awq, tok_awq, "awq", N_TRAJS)

del model_awq, tok_awq
gc.collect(); torch.cuda.empty_cache()
print(f"\nAll done! Trajectories saved to Drive.", flush=True)

In [ ]:
# Verify
for label in ["bf16", "awq"]:
    d = os.path.join(DRIVE_BASE, f"trajs_3b_{label}")
    files = sorted(glob.glob(os.path.join(d, "batch_*.pt")))
    total = sum(torch.load(f, map_location="cpu")["hidden_seqs"].shape[0] for f in files)
    print(f"{label}: {len(files)} batch files, {total} trajectories", flush=True)